In [1]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string

from collections import Counter

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

from airouter import AiRouter

client = AiRouter(
   api_key="sk-eXC2ZNKrhHs-9T2Ei1LjOA",
)

In [2]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [3]:
pos_tags = []
pos_tag_story_begin = []
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        if i == 0:
            pos_tag_story_begin.append(first_word.pos_)

        pos_tags.append(first_word.pos_)

In [4]:
pos_counter = Counter(pos_tag_story_begin)

In [8]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(token.lemma_)
        elif token.pos_ == 'VERB':
            verbs.add(token.lemma_)
        elif token.pos_ == 'ADJ':
            adjectives.add(token.lemma_)

In [9]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [5]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [ ]:
story_features = ['dialoog', 'slecht einde', 'plot twist', 'voorspelling', 'conflict']

In [7]:
def generate_prompt(chosen_pos_tag):
    prompt = f"""Vertel een verhaal. 
Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

    return prompt

system_propmt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de 4 en 6 en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

In [9]:
for pos_tag in tqdm(['ADV', 'PRON']):
    prompt = generate_prompt(pos_tag)

    for _ in range(5):
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_propmt},
                {"role": "user", "content": prompt},
            ],
            models=["llama-3.3-70b"]
        )

        completion = response.choices[0].message.content.strip()
        print(pos_tag)
        print(completion)

  0%|          | 0/2 [00:00<?, ?it/s]

ADV
Een keer was ik met mijn hond in het park. Hij heet Fikkie: een vrolijke labrador. Hij rende en rende maar, en ging zo snel dat ik hem niet meer kon bijhouden. Toen viel Fikkie in het water en werd helemaal nat! Ik moest heel hard lachen. Fikkie keek me ook aan alsof hij zei: "Wat lach je nu? Ik ben net nat geworden!" Toen ging Fikkie schudden, en toen spetterde al het water op me! Ik werd helemaal nat! Ik moest nog harder lachen. Fikkie vond het ook heel leuk. Samen gingen we schudden en werden we allebei nog natter! We hadden zoveel lol, en toen moesten we naar huis. Thuis kregen we warme chocolademelk en droge kleren. Fikkie kreeg een warm bad. We waren allebei moe maar blij, en we konden niet wachten tot we weer zoiets konden doen. Ik hou van Fikkie, hij is de beste hond van de hele wereld!
ADV
Soms ga ik naar de dierentuin. 
Ik vind olifanten en giraffen echt zo leuk, die zijn zo groot! 
Mijn vader zei dat we een keer met de trein naar de dierentuin kunnen gaan. 
Ik mag dan vo

 50%|█████     | 1/2 [01:11<01:11, 71.76s/it]

ADV
Vals liep ik naar buiten en toen zag ik een enorme draak staan. Hij was groen en had grote scherpe tanden. Mijn mama zei dat ik niet bang hoefde te zijn, want hij was een vriendelijke draak. Hij heette "Dragon" en hij kon praten! Dragon zei dat hij op zoek was naar vrienden om mee te spelen. Ik was zo opgewonden! We gingen samen naar het park en speelden "verstoppen". Dragon was zo groot dat hij zich overal kon verstoppen. Ik moest heel goed zoeken om hem te vinden. Toen we moe waren, gingen we naar huis en aten we samen een ijsje. Dragon hield van ijs met sprinkels en ik ook! Nu ben ik elke dag met Dragon naar het park. We hebben zoveel plezier samen. En ik ben niet meer bang voor hem, want hij is mijn beste vriend.
PRON
Hij was een grote beer. Hij had een rode neus en een groene hoed. Hij liep door het bos, waar hij veel vrienden had. Zijn beste vrienden waren een eekhoorn, een vogel en een konijn. Hij at altijd honing. Hij kocht het bij de imker, zijn buurman. De imker had zo ve

100%|██████████| 2/2 [01:57<00:00, 58.67s/it]

PRON
Mijn vader heeft een grote jungleboot, hij neemt ons vaak mee op avontuur. 
Hij vertelde me dat hij hem heeft gekregen van een vriend, in ruil voor een zak snoep. 
Ik vond dat zo cool! 
Mijn vader zei dat hij binnenkort weer een tocht gaat maken, hij vroeg of ik mee wilde gaan. 
Ik zei: "Ja, wat supergaaf!" 
Hij lachte en zei: "Dan gaan we samen op avontuur, jongen!" 
Ik kan niet wachten totdat hij weer vertrekt!
